# CUDA coverage-pattern mining with cuCMine

Run with a CUDA GPU and CuPy installed. Import the local cuPAMI checkout or install `cupami` after this algorithm is released. Parquet examples additionally require PyArrow.


In [ ]:
from cupami import cuCMine

rows = [["a", "b"], ["a"], ["a"], ["b"], ["c"]]
miner = cuCMine(rows, minRF=0.2, minCS=0.8, maxOR=0.5)
miner.mine()
patterns = miner.getPatterns()
assert patterns == {"a\tb": 4, "a\tb\tc": 5, "a\tc": 4}
patterns


All thresholds are ratios. Each item must meet `minRF`; union coverage must meet `minCS`. Items are ordered by descending support with lexical ties. At each extension, overlap with the prefix union divided by the added item’s support must be at most `maxOR`. Low-coverage prefixes continue growing. Output values are absolute union counts. Empty rows are omitted and duplicates count once.


In [ ]:
miner.getPatternsAsDataFrame()


In [ ]:
print("Loading (seconds):", miner.getLoadingTime())
print("Mining (seconds):", miner.getMiningTime())
print("Total (seconds):", miner.getTotalRuntime())


The same miner accepts CSV, BIN and Parquet paths. This example writes its dataset and output into a temporary directory.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from cupami import convertFile

with TemporaryDirectory() as directory:
    source = Path(directory) / "transactions.bin"
    convertFile([[0, 1], [0], [0], [1], [2]], source)
    file_miner = cuCMine(source, minRF=0.2, minCS=0.8, maxOR=0.5, use_gds=False)
    file_miner.mine()
    output = Path(directory) / "patterns.txt"
    file_miner.save(output)
    print(output.read_text())
